# Spam Detection - Model Training with CountVectorizer

This notebook uses basic NLP preprocessing and classical machine learning only.

Models compared:
- Multinomial Naive Bayes
- Logistic Regression
- Linear SVC

The best model is selected using F1 score.


## 1. Import Required Libraries

In [1]:
import pandas as pd
import nltk
import string
import pickle

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


## 2. Download NLTK Resources

In [2]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Satvik\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Satvik\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Satvik\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## 3. Load the Dataset

Change the path only if your dataset is stored elsewhere.


In [4]:
df = pd.read_csv(r"C:/Users/Satvik/Desktop/MLproject/notebook/data/spam.csv", encoding="latin-1")
df.head()


,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## 4. Inspect Dataset

In [5]:
print(df.columns)
print(df.shape)
df.info()


Index(['Category', 'Message'], dtype='object')
(5572, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  5572 non-null   object
 1   Message   5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


## 5. Text Preprocessing

The messages are lowercased, tokenized, cleaned, stripped of stopwords, and stemmed.


In [6]:
ps = PorterStemmer()
english_stopwords = set(stopwords.words("english"))

def transform_text(text):
    text = str(text).lower()
    tokens = nltk.word_tokenize(text)

    tokens = [word for word in tokens if word.isalnum()]

    tokens = [
        word for word in tokens
        if word not in english_stopwords
        and word not in string.punctuation
    ]

    tokens = [ps.stem(word) for word in tokens]

    return " ".join(tokens)


## 6. Apply Text Transformation

In [7]:
df["transformed_text"] = df["Message"].apply(transform_text)
df[["Message", "transformed_text"]].head()


,Message,transformed_text
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


## 7. Encode Ham and Spam Labels

In [8]:
label_encoder = LabelEncoder()

df["target"] = label_encoder.fit_transform(df["Category"])

print(
    dict(
        zip(
            label_encoder.classes_,
            label_encoder.transform(label_encoder.classes_)
        )
    )
)

df[["Category", "target"]].head()


{'ham': np.int64(0), 'spam': np.int64(1)}


,Category,target
0,ham,0
1,ham,0
2,spam,1
3,ham,0
4,ham,0


## 8. Split Features and Target

In [9]:
X = df["transformed_text"]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 4457
Testing samples: 1115


## 9. Count Vectorization

`CountVectorizer` converts each cleaned SMS into numerical word-count features.

The vocabulary is learned only from the training data. The test data uses `transform()` to avoid data leakage.


In [10]:
vectorizer = CountVectorizer(max_features=3000)

X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

print("Training feature shape:", X_train_vectorized.shape)
print("Testing feature shape:", X_test_vectorized.shape)


Training feature shape: (4457, 3000)
Testing feature shape: (1115, 3000)


## 10. Define Classical ML Models

In [11]:
models = {
    "Multinomial Naive Bayes": MultinomialNB(),

    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Linear SVC": LinearSVC(
        random_state=42
    )
}

models


{'Multinomial Naive Bayes': MultinomialNB(),
 'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
 'Linear SVC': LinearSVC(random_state=42)}

## 11. Train and Evaluate Models

We calculate accuracy, precision, recall, and F1 score for every model.


In [12]:
model_report = {}

for model_name, model in models.items():
    model.fit(X_train_vectorized, y_train)

    y_pred = model.predict(X_test_vectorized)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    model_report[model_name] = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }

    print(f"\n{model_name}")
    print("-" * 50)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")



Multinomial Naive Bayes
--------------------------------------------------
Accuracy : 0.9803
Precision: 0.9504
Recall   : 0.8993
F1 Score : 0.9241

Logistic Regression
--------------------------------------------------
Accuracy : 0.9812
Precision: 0.9848
Recall   : 0.8725
F1 Score : 0.9253

Linear SVC
--------------------------------------------------
Accuracy : 0.9839
Precision: 0.9645
Recall   : 0.9128
F1 Score : 0.9379


## 12. Compare Model Performance

In [13]:
model_results = pd.DataFrame(model_report).T

model_results = model_results.sort_values(
    by="f1_score",
    ascending=False
)

model_results


,accuracy,precision,recall,f1_score
Linear SVC,0.983857,0.964539,0.912752,0.937931
Logistic Regression,0.981166,0.984848,0.872483,0.925267
Multinomial Naive Bayes,0.980269,0.950355,0.899329,0.924138


## 13. Select the Best Model Using F1 Score

F1 score balances precision and recall and is more informative than accuracy alone for spam detection.


In [14]:
best_model_name = max(
    model_report,
    key=lambda model_name: model_report[model_name]["f1_score"]
)

best_model = models[best_model_name]
best_f1_score = model_report[best_model_name]["f1_score"]

print("Best Model:", best_model_name)
print("Best F1 Score:", best_f1_score)


Best Model: Linear SVC
Best F1 Score: 0.9379310344827586


## 14. Detailed Evaluation of the Best Model

In [15]:
best_y_pred = best_model.predict(X_test_vectorized)

print("Classification Report:\n")
print(
    classification_report(
        y_test,
        best_y_pred,
        target_names=label_encoder.classes_
    )
)

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, best_y_pred))


Classification Report:

              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       966
        spam       0.96      0.91      0.94       149

    accuracy                           0.98      1115
   macro avg       0.98      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115

Confusion Matrix:

[[961   5]
 [ 13 136]]


## 15. Save the Best Model and CountVectorizer

`wb` creates the `.pkl` files if they do not already exist.


In [16]:
model_path = r"C:/Users/Satvik/Desktop/MLproject/artifacts/model.pkl"
preprocessor_path = r"C:/Users/Satvik/Desktop/MLproject/artifacts/preprocessor.pkl"

with open(model_path, "wb") as file:
    pickle.dump(best_model, file)

with open(preprocessor_path, "wb") as file:
    pickle.dump(vectorizer, file)

print("Model saved at:", model_path)
print("CountVectorizer saved at:", preprocessor_path)


Model saved at: C:/Users/Satvik/Desktop/MLproject/artifacts/model.pkl
CountVectorizer saved at: C:/Users/Satvik/Desktop/MLproject/artifacts/preprocessor.pkl


## 16. Test the Model on a New SMS

In [17]:
sample_message = "Congratulations! You have won a free prize. Call now to claim."

transformed_message = transform_text(sample_message)

vectorized_message = vectorizer.transform(
    [transformed_message]
)

prediction = best_model.predict(
    vectorized_message
)[0]

predicted_label = label_encoder.inverse_transform(
    [prediction]
)[0]

print("Message:", sample_message)
print("Prediction:", predicted_label)


Message: Congratulations! You have won a free prize. Call now to claim.
Prediction: spam


## Conclusion

In this notebook:

- SMS text was cleaned using basic NLP preprocessing.
- `CountVectorizer` converted messages into word-count features.
- Multinomial Naive Bayes, Logistic Regression, and Linear SVC were compared.
- Accuracy, precision, recall, and F1 score were evaluated.
- The best model was selected using F1 score.
- The best classifier and fitted CountVectorizer were saved as pickle files.

No deep learning model or deep learning framework is used.
